In [23]:
"""
- 输入：把 30x30 迷宫展平成长度 900 的一维向量（每个格子映射到 0~4）
- 网络：900 -> 4096 -> 512 -> 4（ReLU）
- 训练：用 train_data.csv + train_answer.csv 做回归（MSE），Adam，小批量（PyTorch）
- 输出：对 test_data.csv 预测，写 result.csv（每行 4 个实数）

用法：
    python baseline.py train_data.csv train_answer.csv test_data.csv result.csv
"""

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import zipfile
import os
import random
from sklearn.ensemble import RandomForestRegressor

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

N = 30
D = N * N

# ===== 数据读取与编码 =====

def encode_lines(path: str) -> np.ndarray:
    char_id = {".": 0, "#": 1, "?": 2, "S": 3, "T": 4}
    with open(path, "r", encoding="utf-8-sig") as f:
        xs = [[char_id[c] for c in line.strip()] for line in f if line.strip()]
    return np.asarray(xs, dtype=np.float32)

def read_y(path: str) -> np.ndarray:
    return np.loadtxt(path, delimiter=",", dtype=np.float32, encoding="utf-8-sig")

# ===== 结果写出 =====

def write_result(path: Path, pred: np.ndarray) -> None:
    np.savetxt(path, pred, delimiter=",", fmt="%.6f")

# ===== 模型训练 =====
from scipy.ndimage import label
from collections import deque

def extract_features(maze: np.ndarray) -> np.ndarray:
    features = maze.flatten().tolist()

    # 1. 整体的空地数量，障碍数量，问号数量
    empty_count = np.sum(maze == 0)
    obstacle_count = np.sum(maze == 1)
    unknown_count = np.sum(maze == 2)
    features.extend([empty_count, obstacle_count, unknown_count])

    # 2. 每一行，每一列的空地障碍问号数量
    for i in range(N):
        features.extend([
            np.sum(maze[i] == 0),  # 行空地数量
            np.sum(maze[i] == 1),  # 行障碍数量
            np.sum(maze[i] == 2)   # 行问号数量
        ])
        features.extend([
            np.sum(maze[:, i] == 0),  # 列空地数量
            np.sum(maze[:, i] == 1),  # 列障碍数量
            np.sum(maze[:, i] == 2)   # 列问号数量
        ])

    # 3. S到T的最短路和连通块特征
    start = tuple(np.argwhere(maze == 3)[0])  # S的位置
    target = tuple(np.argwhere(maze == 4)[0])  # T的位置

    def bfs(start, target):
        queue = deque([start])
        visited = set()
        visited.add(start)
        distance = 0
        while queue:
            for _ in range(len(queue)):
                x, y = queue.popleft()
                if (x, y) == target:
                    return distance
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < N and 0 <= ny < N and (nx, ny) not in visited and maze[nx, ny] != 1:
                        visited.add((nx, ny))
                        queue.append((nx, ny))
            distance += 1
        return float('inf')  # 如果没有路径

    def bfs2(start, maze):
        queue = deque([start])
        visited = set()
        visited.add(start)
        reachable_count = 0
        while queue:
            for _ in range(len(queue)):
                x, y = queue.popleft()
                reachable_count += 1  # 统计可达空地数量
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < N and 0 <= ny < N and (nx, ny) not in visited and maze[nx, ny] != 1:
                        visited.add((nx, ny))
                        queue.append((nx, ny))
        return reachable_count  # 返回可达空地数量

    shortest_path = bfs(start, target)

    # 计算连通块
    def count_connected_components(maze):
        labeled, num_features = label(maze == 0)  # 只考虑空地
        return num_features

    connected_components = count_connected_components(maze)

    features.extend([shortest_path, connected_components])
    features.append(bfs2(start, maze))

    # 4. 将所有问号视为障碍后的特征
    maze_with_obstacles = np.where(maze == 2, 1, maze)  # 将问号视为障碍
    connected_components_with_obstacles = count_connected_components(maze_with_obstacles)
    features.append(connected_components_with_obstacles)
    features.append(bfs2(start, maze_with_obstacles))

    # 5. 每一个格子周围的5x5格子中的空地数量，障碍数量，问号数量
    for i in range(N):
        for j in range(N):
            local_area = maze[max(0, i-2):min(N, i+3), max(0, j-2):min(N, j+3)]
            local_empty = np.sum(local_area == 0)
            local_obstacle = np.sum(local_area == 1)
            local_unknown = np.sum(local_area == 2)
            features.extend([local_empty, local_obstacle, local_unknown])

    return np.array(features)


def train_model(train_x_path: str, train_y_path: str) -> RandomForestRegressor:
    # 读取训练数据
    x_train = encode_lines(train_x_path)
    y_train = read_y(train_y_path)
    print("feature")

    # 提取特征
    feature_list = []
    for maze in x_train:
        features = extract_features(maze.reshape(N, N))  # 将一维向量重塑为二维迷宫
        feature_list.append(features)

    feature_array = np.array(feature_list)

    # 训练随机森林模型
    print("train")
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(feature_array, y_train)
    return model

# ===== 预测 =====

def post_process(pred: np.ndarray, maze: np.ndarray) -> np.ndarray:
    # 计算障碍数量和问号数量
    obstacle_count = np.sum(maze == 1)
    unknown_count = np.sum(maze == 2)

    # 对每个预测结果进行后处理
    y1 = np.clip(np.round(pred[0]), obstacle_count, obstacle_count + unknown_count)
    y2 = np.round(pred[1])
    y3 = np.round(pred[2])
    y4 = np.round(pred[3])

    return np.array([y1,y2,y3,y4])

def predict(model: RandomForestRegressor, test_x_path: str) -> np.ndarray:
    x_test = encode_lines(test_x_path)

    # 提取特征
    feature_list = []
    for maze in x_test:
        features = extract_features(maze.reshape(N, N))  # 将一维向量重塑为二维迷宫
        feature_list.append(features)

    feature_array = np.array(feature_list)

    pred = model.predict(feature_array)
    res = []
    for maze, p in zip(x_test, pred):
        res.append(post_process(p,maze))
    return np.array(res)


# ===== 主流程 =====


TRAIN_PATH = "/bohr/train-abk9/v1/"  # 训练集路径


# 训练集
train_x_path = TRAIN_PATH + "train_data.csv"
train_y_path = TRAIN_PATH + "train_answer.csv"



out_path = Path("result.csv")
epochs = 3

model = train_model(train_x_path, train_y_path)



In [4]:
if os.environ.get("DATA_PATH"):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"  # 测试集路径
else:
    DATA_PATH = "/bohr/mazeval-7zx2/v1/"  # 本地测试回退

# 测试集
testA_path = DATA_PATH + "val_data.csv"
testB_path = DATA_PATH + "test_data.csv"

#分别预测
pred_A = predict(model, testA_path)
pred_B = predict(model, testB_path)

#合并预测结果

submissionA = pd.DataFrame(pred_A)
submissionA.to_csv("./submission_val.csv", index=False, header=False)

submissionB = pd.DataFrame(pred_B)
submissionB.to_csv("./submission_test.csv", index=False, header=False)

files_to_zip = ['./submission_val.csv', './submission_test.csv']
zip_filename = 'submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created succefully!')